# LEA NJSLA ISR Batch PDF Splitting

The goal of this python script is to split NJSLA ISRs and create a two page pdf and name the PDF with a student's State ID within a folder containing multiple ISR files. The following packages were used to complete this task:

 Creating a script to split PDFs and assingn the local id as the file name for the ouput. The following packages were used below:
- [pdfplumber](https://pypi.org/project/pdfplumber/)
- [pypdf](https://pypi.org/project/pypdf/)
- [tqdm](https://pypi.org/project/tqdm/)
- [pathlib](https://pypi.org/project/pathlib/)

This script focuses on extracting text via a searching for a substring on the document

In [2]:
from pathlib import Path
import pdfplumber as pdfp
import pypdf
from pypdf import PdfReader,PdfWriter
import string as str
import pandas as pd
import numpy as np
from tqdm import tqdm

In [3]:
# importing tabular data
df = pd.read_excel(r"C:\Users\togarro\Documents\NJSLA_ISR_Split\SID_Other_ID_Export.xlsx") #--> file wuith SIDs and Local IDs

In [4]:
# creating SID len variable
df['State ID'] = df['State ID'].astype('string') #--> casting column data as a string
df['Local ID'] = df['Local ID'].astype('string')
sid_len = len(df['State ID'][0]) #--> len of SID

In [5]:
# variables
search_text = 'ID: ' #--> substring search
search_text_len = len(search_text) #--> len object of the substring
sid = [] #--> creating an empty list

In [6]:
# creating folder_path variables
folder_path = Path(r"C:\Users\togarro\NJSLA_ISR_Script_Exports\SY_24_NJSLA_MAT_ISR")

# for loop leveraging iterdir() method to iterate over files in folder
for file_path in folder_path.iterdir():
    pdf_object = f"{file_path}"
    reader = PdfReader(pdf_object)
    writer = PdfWriter() #--> instantiating PDF Writer
    doc_len = len(reader.pages)//2 #--> returning number of pages, divided by 2 to account for 2 page documents
    
    # opening pdf with pdfplumber and searching for text using a for loop
    with pdfp.open(pdf_object) as pdf:
        for pages in tqdm(pdf.pages[::2],desc = 'Retrieving SIDs'): #--> extracting text from every other page and using tqdm to track progress
            text = pages.extract_text_simple() #--> creating string object
            start = text.find(search_text) #--> assigning the index of the search text to start variable
            end = start +  search_text_len + sid_len #--> assigning the index to stop extracting text to the end variable
            sid.append(text[start:end].split(':')[1].strip()) #--> appening and manipulating extracted text from PDF file
    

Retrieving SIDs: 100%|██████████| 2/2 [00:01<00:00,  1.72it/s]


In [7]:
# create a dataframe for extracted SIDs
sid_df = pd.DataFrame({'State ID':sid})

# reseting index
sid_df = sid_df.reset_index(drop = True)

In [8]:
# inner merging dfs to return local ids
sid_merged_df = sid_df.merge(df, on = 'State ID', how = 'inner')

# qa
qa = sid_df[~sid_df['State ID'].isin(df['State ID'])].shape[0]

In [9]:
print(f'There are {qa} SIDs missing from the merge file')

There are 0 SIDs missing from the merge file


In [10]:
# creating index variable
i = 0

# for loop leveraging iterdir() method to iterate over files in folder
for file_path in folder_path.iterdir():
    pdf_object = f"{file_path}"
    reader = PdfReader(pdf_object)

    if qa == 0:
        # error handling
        try:
            for page_num in tqdm(range(0,len(reader.pages),2),desc = 'Spltting ISRs'):
                writer = PdfWriter() #--> resets the writer object in each iteration of the loop
                page_name = sid_merged_df['Local ID'].iloc[i]  #--> returns other id based on indexing of df based on 
                writer.add_page(reader.pages[page_num]) #--> adding first page to PDF export
                if page_num <= len(reader.pages): #--> conditional to ensure that that page num is not outside of index
                     writer.add_page(reader.pages[page_num +1]) #--> adding second page to PDF export
                with open(f'/Users/togarro/NJSLA_ISR_Script_Exports/SY_24_MATH/{page_name}.pdf','wb') as f: #--> file writing
                        writer.write(f)                
                i+=1 #--> adding 1 to the i variable after each iteration
        
        # error description
        except Exception as e:
            print("An exception occurred:", type(e).__name__)

Spltting ISRs: 100%|██████████| 2/2 [00:00<00:00, 14.21it/s]
